In [ ]:
import glob
import os
import h5py
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400 # User can set this outside the class if needed

from astropy.io import fits
from astropy.io.votable import parse_single_table
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.coordinates import SkyCoord
import astropy.units as u

import os
import h5py
from tqdm import tqdm
from multiprocessing import Pool 
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
import numpy as np

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
import astropy.units as u
from astropy.table import Table

from reproject import reproject_interp
from reproject import reproject_exact
from reproject import reproject_adaptive

from scipy.sparse import coo_matrix
from scipy.sparse.linalg import lsqr
import sys 
import gc 
from functools import partial
from IPython.display import clear_output

from SelfCal.MapHelper import bit_to_bool, make_weight, find_outliers, map_pixels
from SelfCal.WCSHelper import load_from_fits, save_to_fits, find_optimal_frame
from SelfCal import MakeMap
from SelfCal.SPHERExUtility import make_fiducial_chunk_map, make_fiducial_chunk_mask, interpolate_array, load_calibration, interp_2d_vertical, interp_1d
import importlib
importlib.reload(MakeMap)

from datetime import datetime
import pandas as pd


### Inspect Exposures

In [ ]:
def plot_map(map, wcs=None, lowp=0.1, highp=95, fig=None, ax=None, colorbar=True, low=None, high=None, cmap='viridis', norm=LogNorm,
            cbar_label='MJy/sr'):
    if low is None or high is None:
        high, low = np.nanpercentile(map[map>0], [highp, lowp])
    if fig is None:
        fig = plt.figure(figsize=(10, 10))
    if ax is None:
        ax = fig.add_subplot(111, projection=wcs)
    if norm is LogNorm:
        im = ax.imshow(map, norm=LogNorm(vmin=low, vmax=high), origin='lower', cmap=cmap)
    else:
        im = ax.imshow(map, vmin=low, vmax=high, origin='lower', cmap=cmap)

    if wcs is not None:
        # Explicitly set axis labels
        ax.coords['ra'].set_axislabel('RA')
        ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
        ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
        ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
        ax.coords['dec'].set_axislabel('DEC')
        ax.grid(color='black', linestyle='--', alpha=0.5)

    if colorbar:
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
        cbar.set_label(cbar_label)
    return fig, ax

In [ ]:
detector = 1
exposure_list = glob.glob(f'/data1/SPHEREx/reproc_data/deep_north/*/*/*/*D{detector}*.fits')
hdul = fits.open(exposure_list[1000])
data = hdul[1].data
wcs = WCS(hdul[1].header, naxis=2)

In [ ]:
# mask = bit_to_bool(hdul[2].data, ignore_list=[21], invert=True)
# masked_data = np.where(mask, data, np.nan)
# plt.imshow(masked_data, norm=LogNorm(vmin=0.1,vmax=0.2))

In [ ]:
chunk_map = make_fiducial_chunk_map(detector, interp_factor=5)
chunk_valid_mask = make_fiducial_chunk_mask([1], interp_factor=5)

In [ ]:
# # Do plot_map(data) and imshow(chunk_map) side by side
# fig, axes = plt.subplots(1, 2, figsize=(15, 7))
# axes[0].imshow(chunk_map, origin='lower', cmap='gray', alpha=0.5)
# plot_map(data, lowp=5, highp=95, ax=axes[1], colorbar=False)


In [ ]:
hdul.info()

In [ ]:
num_subchannels=10
num_channels=17
num_exposures = 100

pixval_med = []
pixval_hi = []
pixval_low = []
for detector in [1, 2, 3, 4, 5, 6]:
    print(f"Processing detector {detector}")
    exposure_list = glob.glob(f'/data1/SPHEREx/reproc_data/deep_north/*/*/*/*D{detector}*.fits')
    det_BC, det_BW = load_calibration(band=detector, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')
    print(f"Generating chunk map")
    chunk_map = make_fiducial_chunk_map(detector, det_BC, num_subchannels=num_subchannels, num_channels=num_channels,
                    channel_file='/home/thomasli/spherex/spherex_channels.csv')
        
    all_data = []
    all_masks = []
    print(f"Loading exposures")
    sub_list = [exposure_list[i] for i in np.random.choice(len(exposure_list), size=num_exposures, replace=False)]
    for exp in tqdm(sub_list):
        with fits.open(exp) as hdul:
            all_data.append(hdul[1].data)
            mask = bit_to_bool(
                hdul[2].data,
                ignore_list=[21],
                bitmask_header=hdul[2].header,
                invert=True
            )
            all_masks.append(mask)
    stacked_data = np.stack(all_data)
    stacked_masks = np.stack(all_masks)

    print(f"Processing chunks")
    for ch in tqdm(range(1, num_channels*num_subchannels+1)):
        chunk_mask = (chunk_map == ch)

        if not np.any(chunk_mask):
            pixval_med.append(np.nan)
            pixval_hi.append(np.nan)
            pixval_low.append(np.nan)
            continue

        pixels_in_chunk = stacked_data[:, chunk_mask]
        masks_in_chunk = stacked_masks[:, chunk_mask]

        valid_pixels = np.where(masks_in_chunk, pixels_in_chunk, np.nan)

        exp_mean = np.nanmean(valid_pixels, axis=1)

        pixval_med.append(np.nanmedian(exp_mean))
        pixval_hi.append(np.nanpercentile(exp_mean, 84))
        pixval_low.append(np.nanpercentile(exp_mean, 16))
    
    clear_output(wait=True)

### Inspect Reprojected Frames

In [ ]:
reproj_sample = MakeMap.load_reproj_file('/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/reprojected/exp_2195_det_00.h5', 
                         fields=['sub_data', 'file_path'])

In [ ]:
file_path = '/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/reprojected/exp_2195_det_00.h5'

In [ ]:
fields=['sub_data', 'file_path']
with h5py.File(file_path, 'r') as file:
    for key in fields:
        if key in ('sub_wcs', 'det_wcs'):
            header_key = 'sub_header' if key == 'sub_wcs' else 'det_header'
            header_str = file[header_key][()].decode('utf-8')
            data[key] = WCS(fits.Header.fromstring(header_str))
        else:
            data[key] = file[key][()]  # Efficient read

In [ ]:
# hdul = fits.open(reproj_sample['file_path'].decode('ascii'))
# data = hdul[1].data
# plot_map(data, lowp=5, highp=95, colorbar=False)

In [ ]:
# plot_map(reproj_sample['sub_data'], lowp=5, highp=95, colorbar=False)

In [ ]:
coords, data, weight, _  = MakeMap._prep_subframe(
    '/home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/reprojected/exp_0000_det_00.h5', None, None, 0, 0, 
                False, True, chunk_map, chunk_valid_mask)

In [ ]:
# plot_map(data*(weight>0), lowp=5, highp=95, colorbar=False)

In [ ]:
med_vals = []
mean_wavs = []
for ch in tqdm(range(15, 35)):
    fit_hdul = fits.open(f'/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_34channels_D4_Ch{ch}_unnormoffsets.fits')
    mosaic = fit_hdul[5].data
    wav = fit_hdul[7].data
    med_val = np.median(mosaic[np.nonzero(mosaic)])
    mean_wav = np.mean(wav[np.nonzero(mosaic)])
    med_vals.append(med_val)
    mean_wavs.append(mean_wav)
med_vals = np.array(med_vals)
mean_wavs = np.array(mean_wavs)

In [ ]:
ch=24
fit_hdul = fits.open(f'/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_34channels_D4_Ch{ch}_unnormoffsets.fits')
mosaic = fit_hdul[5].data
np.percentile(mosaic[np.nonzero(mosaic)], [0.1, 50, 99.9])+mean_offsets[9]

In [ ]:
mean_offsets = []
for ch in tqdm(range(15, 35)):
    cal_path = f'/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_34channels_D4_Ch{ch}.h5'
    chunk_valid_mask = make_fiducial_chunk_mask([ch], num_channels=34, num_subchannels=10)
    with h5py.File(cal_path, 'r') as file:
        cal_offset = file['O'][()]
    mean_offset = np.mean(cal_offset[:,chunk_valid_mask.astype(bool)])
    mean_offsets.append(mean_offset)
mean_offsets = np.array(mean_offsets)

In [ ]:
ch = 16
cal_path = f'/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_34channels_D4_Ch{ch}.h5'
chunk_valid_mask = make_fiducial_chunk_mask([ch], num_channels=34, num_subchannels=10)
with h5py.File(cal_path, 'r') as file:
    cal_offset = file['O'][()]
mean_offset = np.mean(cal_offset[:,chunk_valid_mask.astype(bool)])

In [ ]:
plt.plot(mean_wavs, med_vals)
plt.show()
plt.plot(mean_wavs, mean_offsets)
plt.show()

### Plot Mosaic

In [ ]:
mask = make_fiducial_chunk_mask([24,25],  num_subchannels=10, num_channels=17*2)
mask[[231, 232, 233, 234, 235]] = 0
mask[[245, 246, 247, 248, 249, 250]] = 0

In [ ]:
np.arange(len(mask))[mask==1]

In [ ]:
detector = 6
channel = 7
# fit_hdul = fits.open(f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/mosaic/nep_6p2arcsec_det{detector}_ch{channel}.fits')
fit_hdul = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_34channels_D4_Ch26.fits')
mosaic = fit_hdul[5].data
wcs = WCS(fit_hdul[1].header, naxis=2)


In [ ]:
# plot_map(mosaic-np.percentile(mosaic[mosaic>0], 1), wcs=wcs, low=5E-3, high=1E-1, colorbar=True)
plot_map(mosaic+0.08, wcs=wcs, lowp=1, highp=95, colorbar=True)

In [ ]:
fit_hdul = fits.open('/data3/thomasli/selfcal/mosaics/mosaic_det1_ch4.fits')
# mean = fit_hdul[7].data

In [ ]:
# high, low = np.nanpercentile(mosaic[mosaic>0], [1, 95])
# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(111, projection=wcs)
# im = ax.imshow(mosaic, norm=LogNorm(vmin=low, vmax=high), origin='lower')

# ax.coords['ra'].set_axislabel('RA')
# ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
# ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
# ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
# ax.coords['dec'].set_axislabel('DEC')
# ax.grid(color='black', linestyle='--', alpha=0.5)

# cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
# cbar.set_label('MJy/sr')
# ax.set_title(f'Detector 1 Channel 16', fontsize=16)
# # plt.savefig('det1_ch16.png', dpi=400)


In [ ]:
# fig = plt.figure(figsize=(10, 10))

# # Calculate the center 20% of the mosaic
# ny, nx = mosaic.shape
# x_start, x_end = int(nx * 0.4), int(nx * 0.6)
# y_start, y_end = int(ny * 0.5), int(ny * 0.7)

# # Crop the mosaic and update the WCS
# mosaic_cropped = mosaic[y_start:y_end, x_start:x_end]
# wcs_cropped = wcs.slice((slice(y_start, y_end), slice(x_start, x_end)))

# ax = fig.add_subplot(111, projection=wcs_cropped)
# im = ax.imshow(mosaic_cropped, norm=LogNorm(vmin=low, vmax=high), origin='lower')

# ax.coords['ra'].set_axislabel('RA')
# ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
# ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
# ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
# ax.coords['dec'].set_axislabel('DEC')
# ax.grid(color='black', linestyle='--', alpha=0.5)

# cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
# cbar.set_label('MJy/sr')
# ax.set_title(f'Detector 2 Channel 5 (20% Zoom)', fontsize=16)



In [ ]:
detector = 1
channel = 12
cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{channel}.h5'

with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]
    S = f['S'][:]
    D = f['D'][:]

skymap = S+np.mean(O)+np.mean(D[np.nonzero(D)])
skymap[S==0] = np.nan

In [ ]:
split1 = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det1_6p2arcsec/mosaic/mosaic_det1_ch10_split1.fits')[5].data
split2 = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det1_6p2arcsec/mosaic/mosaic_det1_ch10_split2.fits')[5].data

In [ ]:
np.std(split1-split2)

In [ ]:
# plt.imshow(split1-split2, vmin=-0.05, vmax=0.05)

### Make Gif

In [ ]:
display_range = []
for detector in tqdm([1,2,3,4,5,6]):
    channel = 7
    fit_hdul = fits.open(f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/mosaic/nep_6p2arcsec_det{detector}_ch{channel}.fits')
    # fit_hdul = fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch12_20band.fits')
    mosaic = fit_hdul[0].data
    wcs = WCS(fit_hdul[0].header, naxis=2)
    display_range.append(np.nanpercentile(mosaic[mosaic>0], [0.01, 99]).tolist())

In [ ]:
from PIL import Image

def make_gif(image_list, output_path, duration=200):
    """
    Create a GIF from a list of images.

    Parameters:
    - image_list: List of file paths to images.
    - output_path: Path to save the output GIF.
    - duration: Duration of each frame in milliseconds.
    """
    frames = [Image.open(img) for img in image_list]
    frames[0].save(output_path, save_all=True, append_images=frames[1:], duration=duration, loop=0)

In [ ]:
display_range[4] = [0.2, 0.6]
display_range[5] = [0.3, 0.7]

In [ ]:
high-low+1E-3

In [ ]:
-low + 1E-3

In [ ]:
tbl = Table.read('/home/thomasli/spherex/spherex_channels.csv')

for det in [5, 6]:
    sub_tbl = tbl[tbl['band'] == det]
    channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
    fine_edges = interpolate_array(channel_edges, interp_factor=2)
    wav_mean = (fine_edges[0:-1]+fine_edges[1:])/2
    for i, channel in enumerate(tqdm(range(1, 18))):
        fit_hdul = fits.open(f'/mnt/md124/thomasli/selfcal/outputs/nep_det{det}_6p2arcsec/mosaic/mosaic_det{det}_ch{channel}.fits')
        mosaic = fit_hdul[5].data
        wcs = WCS(fit_hdul[5].header, naxis=2)
        lam = wav_mean[i]
        valid_mask = mosaic > 0
        # mosaic[valid_mask] += -np.percentile(mosaic[valid_mask], 1) + 1E-3
        # mosaic[valid_mask] = np.clip(mosaic[valid_mask], a_min=1E-3, a_max=None)
        # fig, ax = plot_map(mosaic, wcs=wcs, lowp=1, highp=95, colorbar=False)
        low = np.nanpercentile(mosaic[valid_mask], 5)
        high = np.nanpercentile(mosaic[valid_mask], 95)
        mosaic[valid_mask] += -low + 5E-3
        mosaic[valid_mask] = np.clip(mosaic[valid_mask], a_min=5E-3, a_max=None)

        fig, ax = plot_map(mosaic, wcs=wcs, low=5E-3, high=high-low+5E-3, colorbar=True)
        ax.set_title(f'D{det} Ch{channel} $\\lambda = {lam:.4f} \\mu m$')
        fig.savefig(f'/home/thomasli/spherex/selfcal/figures/mosaics/D{det}_Ch{channel}.png', dpi=200, bbox_inches='tight')
        plt.close(fig)

    image_list = [f'/home/thomasli/spherex/selfcal/figures/mosaics/D{det}_Ch{channel}.png' for channel in range(1,18)]

    output_path = f'/home/thomasli/spherex/selfcal/figures/D{det}_mosaics.gif'
    make_gif(image_list, output_path, duration=300)

In [ ]:
make_gif(image_list, output_path, duration=300)

In [ ]:
fit_hdul = fits.open('/data3/thomasli/selfcal/mosaics/mosaic_det1_ch4.fits')


In [ ]:
tbl = Table.read('/home/thomasli/spherex/spherex_channels.csv')

for det in [4]:
    sub_tbl = tbl[tbl['band'] == det]
    channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
    fine_edges = interpolate_array(channel_edges, interp_factor=2)
    wav_mean = (fine_edges[0:-1]+fine_edges[1:])/2
    for i, channel in enumerate(tqdm(range(1, 18))):
        fit_hdul = fits.open(f'/mnt/md124/thomasli/selfcal/mosaics/mosaic_det{det}_ch{channel}.fits')
        lam_mean = fit_hdul[7].data
        lam_std = fit_hdul[8].data
        wcs = WCS(fit_hdul[7].header, naxis=2)

        fig, ax = plot_map(lam_mean, wcs=wcs, lowp=0, highp=100, colorbar=True, norm=None, cbar_label='$\\mu m$')
        ax.set_title(f'D{det} Ch{channel} $\\lambda$ Mean')
        fig.savefig(f'/home/thomasli/spherex/selfcal/figures/mosaics/D{det}_Ch{channel}_lam_mean.png', dpi=200, bbox_inches='tight')
        plt.close(fig)

        fig, ax = plot_map(lam_std, wcs=wcs, lowp=0, highp=100, colorbar=True, norm=None, cbar_label='$\\mu m$')
        ax.set_title(f'D{det} Ch{channel} $\\lambda$ Standard Deviation')
        fig.savefig(f'/home/thomasli/spherex/selfcal/figures/mosaics/D{det}_Ch{channel}_lam_std.png', dpi=200, bbox_inches='tight')
        plt.close(fig)

    image_list = [f'/home/thomasli/spherex/selfcal/figures/mosaics/D{det}_Ch{channel}_lam_mean.png' for channel in range(1,18)]
    output_path = f'/home/thomasli/spherex/selfcal/figures/D{det}_lam_mean.gif'
    make_gif(image_list, output_path, duration=300)

    image_list = [f'/home/thomasli/spherex/selfcal/figures/mosaics/D{det}_Ch{channel}_lam_std.png' for channel in range(1,18)]
    output_path = f'/home/thomasli/spherex/selfcal/figures/D{det}_lam_std.gif'
    make_gif(image_list, output_path, duration=300)

In [ ]:
image_list = [f'/home/thomasli/spherex/selfcal/figures/mosaics/D{detector}_Ch{channel}.png' for channel in range(1,10)]
output_path = f'/home/thomasli/spherex/selfcal/figures/D{detector}_mosaics.gif'
make_gif(image_list, output_path)

### Plot O

In [ ]:
# def compute_channel_averages(data, chunk_map, mask=None, num_subchannels=10, num_channels=17):
#     mean_values = []
#     for ch in tqdm(range(1, num_channels * num_subchannels + 1)):
#         chunk_mask = (chunk_map == ch)

#         pixels_in_chunk = data[chunk_mask]
#         if mask is not None:
#             masks_in_chunk = mask[chunk_mask]
#             valid_pixels = np.where(masks_in_chunk, pixels_in_chunk, np.nan)
#         else:
#             valid_pixels = pixels_in_chunk

#         data_mean = np.nanmean(valid_pixels)
#         mean_values.append(data_mean)
#     return np.array(mean_values)
def compute_channel_averages(data, chunk_map, mask=None, num_subchannels=10, num_channels=17):
    total_channels = num_channels * num_subchannels
    flat_data = data.ravel()
    flat_indices = np.clip(chunk_map.ravel() - 1, 0, total_channels - 1)

    if mask is not None:
        flat_mask = mask.ravel()
        sum_weights = np.where(flat_mask, flat_data, 0)
        count_weights = flat_mask.astype(int)
    else:
        sum_weights = flat_data
        count_weights = np.ones_like(flat_data, dtype=int)

    sums = np.bincount(
        flat_indices, 
        weights=sum_weights, 
        minlength=total_channels
    )
    
    counts = np.bincount(
        flat_indices, 
        weights=count_weights, 
        minlength=total_channels
    )

    with np.errstate(divide='ignore', invalid='ignore'):
        mean_values = sums / counts

    return mean_values

avg_gain_factor = []
for det in [1, 2, 3, 4, 5, 6]:

    qr = 1
    gain_file = glob.glob(f'/data1/SPHEREx/CalFiles/abs_gain_matrix_qr{qr}/*/{det}/*.fits')[0]
    gain1 = fits.open(gain_file)[1].data

    qr = 2
    gain_file = glob.glob(f'/data1/SPHEREx/CalFiles/abs_gain_matrix_qr{qr}/*/{det}/*.fits')[0]
    gain2 = fits.open(gain_file)[1].data

    gain_factor = gain2/gain1

    det_BC, det_BW = load_calibration(band=det, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')
    chunk_map, lvf_params = make_fiducial_chunk_map(det, det_BC, num_subchannels=10, num_channels=17,
                    channel_file='/home/thomasli/spherex/spherex_channels.csv')

    avg_gain_factor.append(compute_channel_averages(gain_factor, chunk_map))

In [ ]:
import warnings
import numpy as np
import pandas as pd # Make sure pandas is imported
from datetime import timedelta

# Astropy imports
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord, get_sun, EarthLocation
from astropy.utils.exceptions import AstropyWarning

def load_solar_flux_data(filepath):
    """
    Loads and prepares the solar flux data file for interpolation.
    This function is correct and remains unchanged.
    """
    col_names = [
        'fluxdate', 'fluxtime', 'fluxjulian', 'fluxcarrington',
        'fluxobsflux', 'fluxadjflux', 'fluxursi'
    ]
    
    try:
        df_flux = pd.read_csv(
            filepath,
            delim_whitespace=True,
            comment='#',
            header=None,
            skiprows=1,
            names=col_names
        )
        
        # Explicitly convert the columns we need to numeric data types.
        df_flux['fluxjulian'] = pd.to_numeric(df_flux['fluxjulian'], errors='coerce')
        df_flux['fluxobsflux'] = pd.to_numeric(df_flux['fluxobsflux'], errors='coerce')
        
        # Remove any rows where the conversion might have failed.
        df_flux.dropna(subset=['fluxjulian', 'fluxobsflux'], inplace=True)
        
        # Ensure the data is sorted by Julian Date for interpolation.
        df_flux.sort_values('fluxjulian', inplace=True)
        
        print(f"Successfully loaded and cleaned solar flux data from '{filepath}'.")
        return df_flux
    except FileNotFoundError:
        print(f"ERROR: Solar flux data file not found at '{filepath}'.")
        return None


def analyze_airglow_conditions(header, df_solar, satellite_altitude_km=700.0):
    """
    Calculates physical conditions relevant to He I 1.083um airglow intensity.

    This function now also returns the observation time in two formats:
    - A Python datetime object, perfect for time-series plots.
    - The Modified Julian Date (MJD) as a float.
    """
    # --- MODIFIED: Initialize the dictionary with new keys for date info ---
    results = {
        'obs_datetime': None,
        'mjd_avg': np.nan,
        'angle_to_sun_deg': np.nan,
        'angle_from_nadir_deg': np.nan,
        'f107_flux': np.nan,
    }

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", AstropyWarning)

        try:
            # --- Get Time and Pointing Info ---
            obs_time = Time(header['DATE-AVG'])
            
            # --- NEW: Add date information to the results dictionary ---
            # Add the Python datetime object for easy plotting with Matplotlib/Seaborn
            results['obs_datetime'] = obs_time.datetime
            # Add the MJD for numerical analysis or scatter plots vs. time
            results['mjd_avg'] = header.get('MJD-AVG', obs_time.mjd)

            pointing_ra = header['CRVAL1'] * u.deg
            pointing_dec = header['CRVAL2'] * u.deg
            pointing_coord = SkyCoord(ra=pointing_ra, dec=pointing_dec, frame='icrs')

            # --- Calculate Angle to Sun ---
            sun_coord = get_sun(obs_time)
            results['angle_to_sun_deg'] = pointing_coord.separation(sun_coord).to_value(u.deg)

            # --- Calculate Angle from Nadir ---
            sat_lat = header['HIERARCH SGT_LAT_MIDPT'] * u.deg
            sat_lon = header['HIERARCH SGT_LON_MIDPT'] * u.deg
            satellite_location = EarthLocation(
                lat=sat_lat, lon=sat_lon, height=satellite_altitude_km * u.km
            )
            satellite_itrs = satellite_location.get_itrs(obstime=obs_time)
            nadir_coord = SkyCoord(
                x=-satellite_itrs.x, y=-satellite_itrs.y, z=-satellite_itrs.z,
                frame='itrs', obstime=obs_time
            )
            results['angle_from_nadir_deg'] = pointing_coord.separation(nadir_coord).to_value(u.deg)

            # --- Interpolate Solar Flux ---
            obs_jd = obs_time.jd
            interpolated_flux = np.interp(
                obs_jd,
                df_solar['fluxjulian'].values,
                df_solar['fluxobsflux'].values
            )
            results['f107_flux'] = interpolated_flux

        except KeyError as e:
            print(f"Warning: Missing essential header keyword {e}. Some results will be NaN.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

    return results

In [ ]:
df_solar = load_solar_flux_data('misc/fluxtable.txt')
analyze_airglow_conditions(hdr, df_solar)

In [ ]:
detector = 1

reproj_list = glob.glob(f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/reprojected/*.h5')

result_list = []

for f in tqdm(reproj_list):
    exp_path = MakeMap.load_reproj_file(f, fields=['file_path'])['file_path'].decode('ascii')
    exp_hdul = fits.open(exp_path)
    hdr = exp_hdul[1].header
    result = analyze_airglow_conditions(hdr, df_solar)
    result_list.append(result)

In [ ]:
df = pd.DataFrame(result_list)

In [ ]:
def extract_calibration(det, ch):
    cal_path = f'/data1/thomasli/selfcal/outputs/nep_det{det}_6p2arcsec/calibration/cal_det{det}_ch{ch}.h5'

    with h5py.File(cal_path, 'r') as f:
        O = f['O'][:]

    chunk_mask = make_fiducial_chunk_mask([ch],  num_subchannels=10, num_channels=17)
    crop_O = O[:, chunk_mask==1]
    return crop_O

In [ ]:
det_cal = np.hstack([extract_calibration(det, ch) for ch in range(1, 18)])

In [ ]:
det_cal.shape

In [ ]:
cal_med = []
cal_hi = []
cal_low = []
wav_mean = []

tbl = Table.read('/home/thomasli/spherex/spherex_channels.csv')

for det in tqdm(range(1, 7)):
    det_cal = np.hstack([extract_calibration(det, ch) for ch in range(1, 18)])
    if det not in [1, 2,]:
        det_cal *= avg_gain_factor[det-1]
    cal_med.append(np.median(det_cal, axis=0))
    cal_hi.append(np.percentile(det_cal, 84, axis=0))
    cal_low.append(np.percentile(det_cal, 16, axis=0))

    sub_tbl = tbl[tbl['band'] == det]
    channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
    fine_edges = interpolate_array(channel_edges, interp_factor=10)
    wav_mean.append((fine_edges[0:-1]+fine_edges[1:])/2)

In [ ]:
# for i, det in enumerate([1, 2, 3, 4, 5, 6]):
#     # plt.scatter(wav_mean[index_start:index_end], cal_med[index_start:index_end], 
#     #             s=0.5, label=f'Detector {detector}', edgecolors='none', color='black')
#     plt.plot(wav_mean[i], cal_med[i], 
#                 label=f'D{det}', color=f'C{det-1}', linewidth=0.5)
#     plt.fill_between(wav_mean[i], cal_low[i], cal_hi[i], 
#                      alpha=0.3, facecolor=f'C{det-1}')
    
# # plt.yscale('log')
# plt.xlabel('Wavelength [um]')
# plt.ylabel('Calibration Offset [MJy/sr]')
# plt.legend(loc='upper right')
# plt.ylim(0.05,0.7)
# plt.xlim(0.74, 5.0)
# # plt.savefig('/home/thomasli/spherex/selfcal/figures/spherex_offset.png', dpi=200, bbox_inches='tight', transparent=True)


In [12]:
import json

In [15]:
offset_dict = {}
for i, det in tqdm(enumerate(range(1, 7))):
    offset_dict.update({
        f'D{det}': {
            'offset_med': cal_med[i].tolist(),
            'offset_hi': cal_hi[i].tolist(),
            'offset_low': cal_low[i].tolist(),
            'wav_mean': wav_mean[i].tolist()
        }
    })

with open('/home/thomasli/spherex/selfcal/figures/spherex_offset.json', 'w') as f:
    json.dump(offset_dict, f)

6it [00:00, 27533.72it/s]


In [ ]:
# plt.plot(wav_mean, pixval_med, linewidth=0.5, label='Background Level')
# plt.fill_between(wav_mean, pixval_low, pixval_hi, alpha=0.3)
# plt.plot(wav_mean, cal_med, linewidth=0.5, label='Calibration Offset')
# plt.fill_between(wav_mean, cal_low, cal_hi, alpha=0.3)
# plt.legend()

In [ ]:
# plt.plot(wav_mean, pixval_med-cal_med, linewidth=0.5, label='Background Level')

In [ ]:
# plt.scatter(df['obs_datetime'] , O.max(axis=1), s=1, edgecolor='None')

In [ ]:
# _ = plt.plot(np.arange(20), O[:, 146:166][0:100].T)

In [ ]:
np.save('/data1/SPHEREx/CalFiles/qr2gain_over_qr1gain.npy', np.array(avg_gain_factor))

### Compare To Zodi

In [ ]:
sample_idx = np.random.choice(np.arange(3000), size=20, replace=False)


In [ ]:
zodi_med = []
for det in [1, 2, 3, 4, 5, 6]:
    det_BC, det_BW = load_calibration(band=det, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')
    chunk_map = make_fiducial_chunk_map(det, det_BC, num_subchannels=10, num_channels=17,
                    channel_file='/home/thomasli/spherex/spherex_channels.csv')

    reproj_list = sorted(glob.glob(f'/data1/thomasli/selfcal/outputs/nep_det{det}_6p2arcsec/reprojected/*.h5'))
    det_zodi_med = []
    for i in tqdm(sample_idx):
        reproj_file = reproj_list[i]
        result = MakeMap.load_reproj_file(reproj_file, fields=('file_path',))
        hdul = fits.open(result['file_path'].decode('ascii'))
        data = hdul[4].data
        det_zodi_med.append(compute_channel_averages(data, chunk_map))
    zodi_med.append(np.median(np.stack(det_zodi_med), axis=0))

In [ ]:
zodi_med = np.array(zodi_med)

In [ ]:
# for i, det in enumerate([1, 2, 3, 4, 5, 6]):
#     # plt.scatter(wav_mean[index_start:index_end], cal_med[index_start:index_end], 
#     #             s=0.5, label=f'Detector {detector}', edgecolors='none', color='black')
#     plt.plot(wav_mean[i], cal_med[i], 
#                 label=f'D{det}', color=f'C{det-1}', linewidth=0.5)
#     plt.fill_between(wav_mean[i], cal_low[i], cal_hi[i], 
#                      alpha=0.3, facecolor=f'C{det-1}')
#     plt.plot(wav_mean[i], zodi_med[i],#*avg_gain_factor[i], 
#                 color=f'C{det-1}', linewidth=0.5, ls='--')
    
# # plt.yscale('log')
# plt.xlabel('Wavelength [um]')
# plt.ylabel('Calibration Offset [MJy/sr]')
# plt.legend(loc='upper right')
# plt.ylim(0.05,0.7)
# plt.xlim(0.74, 5.0)

In [ ]:
cal_path = f'/data1/thomasli/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{1}.h5'

with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]
    S = f['S'][:]

In [ ]:
# plt.plot(wav_mean[0:170], pixval_med, linewidth=0.5, label='Background Level')
# plt.plot(wav_mean[0:170], pixval_med, linewidth=0.5, label='Background Level')

### Offset vs Time

In [ ]:
det = 2
ch = 5

offset = extract_calibration(det, ch)
reproj_list = sorted(glob.glob(f'/data1/thomasli/selfcal/outputs/nep_det{det}_6p2arcsec/reprojected/*.h5'))


In [ ]:
# ch_offset = np.mean(offset, axis=1)
ch_offset = offset[:, 5]

In [ ]:
exp_idx = 1

hdr_list = []
for reproj_file in tqdm(reproj_list):
    exp_file = MakeMap.load_reproj_file(reproj_file, fields=('file_path',))['file_path'].decode('ascii')
    with fits.open(exp_file) as hdul:
        hdr = hdul[1].header
    hdr_list.append(hdr)

In [ ]:
hdr_list[0]

In [ ]:
def bin_array(data, bin_size):
    n_bins = len(data) // bin_size
    binned_data = data[:n_bins * bin_size].reshape(n_bins, bin_size).mean(axis=1)
    return binned_data

In [ ]:
# sort by mjd
sorted_indices = np.argsort(mjd_list)
sorted_mjd = np.array(mjd_list)[sorted_indices]
sorted_offset = ch_offset[sorted_indices]



In [ ]:
import numpy as np
from scipy.stats import binned_statistic, binned_statistic_2d

bin_width = 0.3
min_mjd = sorted_mjd.min()
max_mjd = sorted_mjd.max()
bins = np.arange(min_mjd, max_mjd + bin_width, bin_width)

bin_means, bin_edges, binnumber = binned_statistic(
    sorted_mjd,      # The array to bin on (your x-axis)
    sorted_offset,   # The values to aggregate (your y-axis)
    statistic='mean',# The statistic you want
    bins=bins        # The bin edges you just created
)

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

In [ ]:
x = [hdr['SPS_ELON'] for hdr in hdr_list]
# plt.scatter(x, ch_offset, s=1, edgecolor='None')
# plt.ylim(0.12, 0.17)


In [ ]:
x = [hdr['SPS_ELON'] for hdr in hdr_list]
y = [hdr['SPS_ELAT'] for hdr in hdr_list]

mean, _, _, _ = binned_statistic_2d(x, y, ch_offset, statistic='mean', bins=100)

### PAH

In [ ]:
def extract_sky(cal_path):
    with h5py.File(cal_path, 'r') as f:
        O = f['O'][:]
        S = f['S'][:]
        D = f['D'][:]

    skymap = S+np.mean(O)+np.mean(D[np.nonzero(D)])
    skymap[S==0] = np.nan
    return skymap

In [ ]:
ch11_data = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_34channels_D4_Ch19.fits')[5].data
ch13_data = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_34channels_D4_Ch30.fits')[5].data
ch12_data = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_34channels_D4_PAH_subchannel225-235_unnormoffsets_widecal.fits')[5].data
wcs = WCS(fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_34channels_D4_PAH_subchannel225-235_unnormoffsets_widecal.fits')[5].header, naxis=2)

# ch11_data = extract_sky('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_det4_ch10.h5')
# ch12_data = extract_sky('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/nep_6p2arcsec_det4_ch12_20band.fits')
# ch13_data = extract_sky('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_det4_ch14.h5')
ch12_sim = (ch11_data + ch13_data)/2


In [ ]:
pah_map = ch12_data-ch12_sim
valid_mask = np.all([ch11_data!=0, ch12_data!=0, ch13_data!=0], axis=0)
pah_map[~valid_mask] = 0

In [ ]:
np.min(pah_map)

In [ ]:
np.nanpercentile(pah_map, 1)

In [ ]:
plot_map(pah_map+np.nanpercentile(pah_map, 1), wcs=wcs, lowp=0.1, highp=80, colorbar=True, norm=None, cbar_label='MJy/sr')
# plt.savefig('figures/PAH_map_new.png', dpi=500, bbox_inches='tight')

In [13]:
ch11_data = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_det4_ch10.fits')[5].data
ch13_data = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_det4_ch14.fits')[5].data
ch12_data = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_det4_ch12.fits')[5].data
wcs = WCS(fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_det4_ch12.fits')[5].header, naxis=2)
lam_std = fits.open('/mnt/md124/thomasli/selfcal/mosaics/mosaic_det4_ch12.fits')[8].data
lam_mean = fits.open('/mnt/md124/thomasli/selfcal/mosaics/mosaic_det4_ch12.fits')[7].data
weight = fits.open('/mnt/md124/thomasli/selfcal/mosaics/mosaic_det4_ch12.fits')[6].data
ch12_sim = (ch11_data + ch13_data)/2

pah_map = ch12_data-ch12_sim
valid_mask = np.all([ch11_data!=0, ch12_data!=0, ch13_data!=0, lam_mean<3.318, lam_mean>3.268], axis=0)
pah_map[~valid_mask] = 0
#lam_mean<3.313, lam_mean>3.273
pah_map[valid_mask] = np.clip(pah_map[valid_mask],a_min=1E-3,a_max=None)

In [ ]:
# fig, ax = plot_map(np.where(valid_mask, lam_mean, np.nan), wcs=wcs, low=3.27, high=3.31, colorbar=True, norm=None, cbar_label='$\\mu m$')
# ax.set_title(f'D{det} Ch{channel} $\\lambda$ Mean')

In [ ]:
pah_map.shape

In [ ]:
center = np.array(np.shape(pah_map))/2#wcs.world_to_pixel_values(270, 66.5)
width = 3000

In [ ]:
# plot_map(pah_map, wcs=wcs, low=1E-3, high=1.5E-2, colorbar=True, cmap='viridis')
# plt.xlim(center[0]-width/2, center[0]+width/2)
# plt.ylim(center[1]-width/2, center[1]+width/2)
# # plt.ylim(51, )
# # plt.savefig('figures/PAH_map.png', dpi=400, bbox_inches='tight')

In [ ]:
# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(111, projection=wcs)
# im = ax.imshow(pah_map, norm=LogNorm(vmin=1e-3, vmax=5e-2), origin='lower')
# ax.set_title('PAH Map')

# ax.coords['ra'].set_axislabel('RA')
# ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
# ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
# ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
# ax.coords['dec'].set_axislabel('DEC')
# ax.grid(color='black', linestyle='--', alpha=0.5)

# cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
# cbar.set_label('MJy/sr')


In [ ]:
cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/nep_6p2arcsec_det4_ch12_20band.fits'
with h5py.File(cal_path, 'r') as f:
    D = f['D'][:]
    O = f['O'][:]
    S = f['S'][:]

### PAH and Other

In [ ]:
map1 = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_3p1arcsec/mosaic/mosaic_det4_ch23-24_narrow.fits')[5].data
map2 = fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_3p1arcsec/mosaic/mosaic_det4_ch26-27_weighted.fits')[5].data
wcs = WCS(fits.open('/mnt/md124/thomasli/selfcal/outputs/nep_det4_3p1arcsec/mosaic/mosaic_det4_ch23-24_narrow.fits')[5].header)


In [ ]:
center = np.array(np.shape(map1))/2#wcs.world_to_pixel_values(270, 66.5)
width = 4000
crop_map1 = map1[int(center[1]-width/2):int(center[1]+width/2), int(center[0]-width/2):int(center[0]+width/2)]
crop_map2 = map2[int(center[1]-width/2):int(center[1]+width/2), int(center[0]-width/2):int(center[0]+width/2)]
ratio_map = crop_map1/crop_map2
valid = ~(np.isnan(ratio_map) | (ratio_map==0))


In [ ]:
mpl.rcParams['figure.dpi'] = 200

In [ ]:
plot_map(crop_map1/crop_map2, wcs=wcs, low=0.9, high=1.2, colorbar=True, cmap='viridis', norm=None)

In [ ]:
heatmap, xedges, yedges = np.histogram2d(crop_map1[valid], ratio_map[valid], bins=200, range=[[0.07, 0.12], [0.95, 1.2]])
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

In [ ]:
from matplotlib.colors import LogNorm, SymLogNorm

In [ ]:
plt.imshow(heatmap.T, extent=extent, origin='lower', aspect='auto')
plt.xlabel('$I_{3.28}$ [MJy/sr]')
plt.ylabel('$\\frac{I_{3.28}}{I_{3.425}}$ Ratio')

### Test Calibration

In [ ]:
det = 1

det_BC, det_BW = load_calibration(band=det, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')
chunk_map, lvf_params = make_fiducial_chunk_map(det, det_BC, num_subchannels=10, num_channels=17,
                   channel_file='/home/thomasli/spherex/spherex_channels.csv')

reproj_list = sorted(glob.glob(f'/data1/thomasli/selfcal/outputs/nep_det{det}_6p2arcsec/reprojected/*.h5'))

det_idx_list = []
exp_idx_list = []
for file in tqdm(reproj_list):
    file_name = os.path.basename(file)
    det_idx_list.append(int(file_name[file_name.find('det_')+4:file_name.find('det_')+6]))
    exp_idx_list.append(int(file_name[file_name.find('exp_')+4:file_name.find('exp_')+8]))

In [44]:
exp_idx = 3000

chs = [i for i in range(1, 18)]
cal = np.array([])
for ch in chs:
    chunk_valid_mask = make_fiducial_chunk_mask([ch], num_subchannels=10)
    cal_path = f'/data1/thomasli/selfcal/outputs/nep_det{det}_6p2arcsec/calibration/cal_det{det}_ch{ch}.h5'
    with h5py.File(cal_path, 'r') as f:
        sub_O = f['O'][:][exp_idx][chunk_valid_mask==1]
    cal = np.concatenate((cal, sub_O))

cal = np.concatenate((cal[:1], cal, cal[-1:]))  # Add endpoints for plotting

In [ ]:
cal_path = '/data1/thomasli/selfcal/outputs/nep_det1_6p2arcsec/calibration/cal_det1_ch15-17_10band_newO.h5'
chunk_valid_mask = make_fiducial_chunk_mask([15, 16, 17], num_subchannels=10)
chunk_valid_mask[141:146] = 0
chunk_valid_mask[166:] = 0

with h5py.File(cal_path, 'r') as f:
    sub_O = f['O'][:][exp_idx][chunk_valid_mask==1]
# cal[chunk_valid_mask==1] = sub_O - np.median(sub_O) + np.median(cal[146:166])

In [ ]:
num_subchannels=10
num_channels=17

pixval_med = []
pixval_hi = []
pixval_low = []

reproj_file = MakeMap.load_reproj_file(reproj_list[exp_idx], fields=('file_path',))
exp_file = reproj_file['file_path'].decode('ascii')

with fits.open(exp_file) as hdul:
    data = hdul[1].data
    mask = bit_to_bool(
        hdul[2].data,
        ignore_list=[],
        invert=True
    )

for ch in tqdm(range(1, num_channels*num_subchannels+1)):
    chunk_mask = (chunk_map == ch)


    pixels_in_chunk = data[chunk_mask]
    masks_in_chunk = mask[chunk_mask]

    valid_pixels = np.where(masks_in_chunk, pixels_in_chunk, np.nan)

    exp_mean = np.nanmean(valid_pixels)

    pixval_med.append(np.nanmedian(exp_mean))
    pixval_hi.append(np.nanpercentile(exp_mean, 84))
    pixval_low.append(np.nanpercentile(exp_mean, 16))

pixval = np.concatenate((np.array([0], ), pixval_med, np.array([0], )))

In [40]:
norm_pixval = pixval - np.min(pixval[pixval!=0])
pixval_map = norm_pixval[chunk_map]
pixval_map = interp_2d_vertical(pixval_map, method='mp')


In [ ]:
# plt.subplot(3,1,1)
# plt.imshow(data[0:400], vmin=0, vmax=0.25)
# plt.axis('off')
# plt.subplot(3,1,2)
# plt.imshow((data - offset_map)[0:400], vmin=0, vmax=0.25)
# plt.axis('off')
# plt.subplot(3,1,3)
# plt.imshow((data - pixval_map)[0:400], vmin=0, vmax=0.25)
# plt.axis('off')

In [ ]:
offset_map = cal[chunk_map]
offset_map = interp_2d_vertical(offset_map, method='mp')

In [51]:
fig, axs = plt.subplots(1, 3, figsize=(10, 4), tight_layout=True)
axs[0].imshow(data, vmin=0.12, vmax=0.15, origin='lower')
axs[0].axis('off')
axs[0].set_title('L2 Frame')
cbar = fig.colorbar(axs[0].images[0], ax=axs[0], orientation='horizontal', fraction=0.046, pad=0.1)
cbar.set_label('MJy/sr')
axs[1].imshow(offset_map, vmin=0.0, vmax=0.3, origin='lower')
axs[1].axis('off')
axs[1].set_title('Offset')
cbar = fig.colorbar(axs[1].images[0], ax=axs[1], orientation='horizontal', fraction=0.046, pad=0.1)
cbar.set_label('MJy/sr')
axs[2].imshow((data - offset_map), vmin=0.0, vmax=0.03, origin='lower')
axs[2].axis('off')
axs[2].set_title('L2 Frame - Offset')
cbar = fig.colorbar(axs[2].images[0], ax=axs[2], orientation='horizontal', fraction=0.046, pad=0.1)
cbar.set_label('MJy/sr')

plt.savefig('/home/thomasli/spherex/selfcal/figures/remove_offset.png', dpi=200, bbox_inches='tight', transparent=True)

In [ ]:
slice_offset_map = offset_map[:, 1020]
slice_pixval_map = pixval_map[:, 1020]
slice_calibrated = data[:, 1020] - offset_map[:, 1020]
slice_data = data[:, 1020]


In [ ]:
# plt.plot(wav_mean[det-1], pixval[1:-1], label='L2 Frame', linewidth=0.5)
# plt.plot(wav_mean[det-1], cal[1:-1], label='Offset', linewidth=0.5)
# plt.plot(wav_mean[det-1], pixval[1:-1]-norm_cal[1:-1], label='L2 Frame - Offset', linewidth=0.5)
# plt.xlabel('Wavelength [um]')
# plt.ylabel('MJy/sr')
# plt.legend()


In [ ]:
detector = 1
det_BC, det_BW = load_calibration(band=detector, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')
chunk_map = make_fiducial_chunk_map(detector, det_BC, num_subchannels=10, num_channels=17,
                   channel_file='/home/thomasli/spherex/spherex_channels.csv')
chunk_valid_mask = make_fiducial_chunk_mask([15, 16, 17], num_subchannels=10)
chunk_valid_mask[141:146] = 0
chunk_valid_mask[166:] = 0

In [30]:
O.shape

In [ ]:
chunk_map

In [36]:
cal_path = '/data1/thomasli/selfcal/outputs/nep_det1_6p2arcsec/calibration/cal_det1_ch15-17_10band_newO.h5'
with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]

exp_idx = 1100
cal = O[exp_idx]
norm_cal = cal - np.min(cal[cal!=0])
# valid_mask = chunk_valid_mask[chunk_map]
offset_map = norm_cal[chunk_map]
offset_map_mp = interp_2d_vertical(offset_map, method='mp')
offset_map_linear = interp_2d_vertical(offset_map, method='linear')

reproj_file = MakeMap.load_reproj_file(reproj_list[exp_idx], fields=('file_path',))
hdul = fits.open(reproj_file['file_path'].decode('ascii'))
data = hdul[1].data

fig, axes = plt.subplots(4, 1, figsize=(10, 12))
axes[0].imshow(((data)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.2))
axes[0].set_title('Raw Exposure')
axes[1].imshow(((data-offset_map)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.2))
axes[1].set_title('Calibrated (Discrete)')
axes[2].imshow(((data-offset_map_mp)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.2))
axes[2].set_title('Calibrated (Mean-Preserving Interpolation)')
axes[3].imshow(((data-offset_map_linear)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.2))
axes[3].set_title('Calibrated (Linear Interpolation)')


In [ ]:
# plt.plot(np.arange(len(pixval)), pixval, linewidth=0.5)
# plt.plot(np.arange(len(cal)), cal, linestyle='--', linewidth=0.5)